# Hotel Booking Cancellation Prediction — EDA & Model Comparison

**Goal:** predict whether a booking will be **cancelled** (`is_canceled`) and output the
**probability** of cancellation.

This notebook covers: data understanding → cancellation-focused EDA → outlier analysis →
preprocessing → linear-vs-non-linear check → comparison of five classifiers
(Logistic Regression, Decision Tree, Random Forest, XGBoost, MLP).

> Heavy lifting lives in `src/`. Saved artefacts land in `reports/` and `models/`.
> Run end-to-end from the CLI with `python run_pipeline.py`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable whether run from notebooks/ or root.
ROOT = Path.cwd()
if (ROOT / "src").exists():
    root = ROOT
else:
    root = ROOT.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from src.data_loader import load_hotel_booking_data, get_default_hotel_booking_path
from src.data_processing import clean_hotel_booking_data, prepare_xy, build_preprocessor, split_data
from src.eda_analysis import iqr_outlier_report

## 1. Data understanding

In [ ]:
df_raw = load_hotel_booking_data(get_default_hotel_booking_path())
print("shape:", df_raw.shape)
df_raw.head()

In [ ]:
# Target distribution — is_canceled
counts = df_raw["is_canceled"].value_counts().sort_index()
print(counts)
print("cancellation rate:", round(df_raw["is_canceled"].mean(), 4))

In [ ]:
# Missing values
miss = df_raw.isnull().sum()
miss[miss > 0].sort_values(ascending=False)

## 2. Data leakage — columns we must drop

`reservation_status` and `reservation_status_date` record the **final outcome** of a
booking (e.g. "Canceled" / "Check-Out"). Using them would let the model cheat and they
are unavailable at prediction time, so `clean_hotel_booking_data` removes them.

In [ ]:
df = clean_hotel_booking_data(df_raw)
print("cleaned shape (leakage cols removed):", df.shape)
[c for c in ["reservation_status", "reservation_status_date"] if c in df.columns]  # -> [] 

## 3. Cancellation-focused EDA

The full set of plots is generated by `run_eda_analysis(df)` into `reports/`. Below we
reproduce the most important patterns inline.

In [ ]:
# Cancellation rate by deposit type — the strongest single driver
(df.groupby("deposit_type")["is_canceled"].mean() * 100).round(1)

In [ ]:
# Cancellation rate by lead-time group (monotonic rise)
bins = [-1, 7, 30, 90, 180, 365, np.inf]
labels = ["0-7", "8-30", "31-90", "91-180", "181-365", "365+"]
grp = pd.cut(df["lead_time"], bins=bins, labels=labels)
(df.groupby(grp, observed=True)["is_canceled"].mean() * 100).round(1).plot(
    kind="bar", color="indianred", edgecolor="black", figsize=(9,4),
    title="Cancellation rate (%) by lead-time group")
plt.ylabel("Cancellation rate (%)"); plt.tight_layout(); plt.show()

In [ ]:
# Generate ALL saved EDA plots into reports/ (class dist, rates by category,
# correlation heatmap, outlier boxplots, missing values, ...)
from src.eda_analysis import run_eda_analysis
run_eda_analysis(df, root / "reports")

## 4. Outlier analysis (IQR — reported, not removed)

In [ ]:
iqr_outlier_report(df)

## 5. Correlation — is the signal linear?

No numeric feature is strongly *linearly* correlated with the target (all |corr| < 0.3),
and the dominant driver (`deposit_type`) is categorical with threshold-like behaviour.
This hints the decision boundary is **non-linear** — confirmed by modeling below.

In [ ]:
num = df.select_dtypes(include=[np.number])
num.corr()["is_canceled"].drop("is_canceled").abs().sort_values(ascending=False).head(8).round(3)

## 6. Preprocessing

- One-hot encode categoricals (rare categories bucketed, unseen ignored).
- **No normalization** (project rule) — numerics pass through; scaling is applied only
  inside the MLP's own pipeline.
- Stratified split using ~80,000 training records.

In [ ]:
X, y = prepare_xy(df)
X_train, X_test, y_train, y_test = split_data(X, y, train_size=80000, random_state=42)
preprocessor = build_preprocessor(X_train)
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)
print(f"train={len(X_train):,}  test={len(X_test):,}  encoded_features={X_train_enc.shape[1]}")

## 7. Train & compare five classifiers

Linear baseline (Logistic Regression) vs non-linear models (Decision Tree, Random Forest,
XGBoost, MLP). Models are selected on **ROC-AUC** because we output a probability.

In [ ]:
from src.modeling import train_and_evaluate, comparison_dataframe, select_best_model
fitted, results = train_and_evaluate(X_train_enc, y_train, X_test_enc, y_test)
comp = comparison_dataframe(results)
comp

In [ ]:
best_name, _ = select_best_model(results, fitted)
print("Best model (ROC-AUC):", best_name)
print(results[best_name]["report"])

## 8. Conclusion

- **Non-linear** problem: every tree/boosted/MLP model beats the linear Logistic
  Regression baseline by a wide margin.
- **XGBoost** gives the best ROC-AUC / F1 and trains fast — selected as the deployed model.
- See `reports/eda_summary.md` and `reports/modeling_concepts.md` for the full write-up
  (bias/variance, metric choice, linear vs non-linear), and `src/api.py` for the FastAPI
  deployment that returns `{prediction, label, cancellation_probability}`.